# Part 4: Model Development
## 4.1 Required Ablation Study

Three configurations for every model type to try:

| Configuration | What it uses |
|---------------|--------------|
| Structured-only | Scaled structured features, no text |
| Text-only | TF-IDF → (optional SVD), no structured features |
| Combined | Both, concatenated |

The ablation demonstrates whether combining modalities was the right call. Sometimes text alone dominates and structured is noise (low signal on structured features); sometimes structured dominates (short or uninformative text); sometimes combining is clearly better. 

In [9]:

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import f1_score, make_scorer
from pathlib import Path


# Load Data
BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "processed" / "train_clean.csv"

df = pd.read_csv(DATA_PATH)

# Target and feature setup

target = "rating"
text_feature = "review_text"

numeric_features = [
    "product_price",
    "seller_rating",
    "delivery_days",
    "product_age_months",
    "num_previous_reviews_by_user",
    "verified_purchase",
    "return_initiated",
    "helpful_votes",
    "discount_pct",
    "product_weight_kg",
    "image_count"
]

categorical_features = [
    "product_category"
]

structured_features = numeric_features + categorical_features

X = df[[text_feature] + structured_features].copy()
y = df[target].copy()

X[text_feature] = X[text_feature].fillna("")


# Cross-validation setup

cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "weighted_f1": make_scorer(f1_score, average="weighted"),
    "macro_f1": make_scorer(f1_score, average="macro")
}


# Preprocessing pipelines

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("svd", TruncatedSVD(
        n_components=100,
        random_state=42
    ))
])


# Required preprocessing configurations

structured_only_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

text_only_preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipeline, text_feature)
    ],
    remainder="drop"
)

combined_preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_pipeline, text_feature),
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

preprocessors = {
    "Structured-only": structured_only_preprocessor,
    "Text-only": text_only_preprocessor,
    "Combined": combined_preprocessor
}

# Model types

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs",
        multi_class="multinomial",
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        dual=False,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=None
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


# Run ablation study

ablation_results = []

for model_name, model in models.items():
    print(f"\nRunning ablation for: {model_name}")

    for config_name, preprocessor in preprocessors.items():
        print(f"  -> Configuration: {config_name}")

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        try:
            scores = cross_validate(
                pipe,
                X,
                y,
                cv=cv_strategy,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
                error_score="raise"
            )

            weighted_scores = scores["test_weighted_f1"]
            macro_scores = scores["test_macro_f1"]

            ablation_results.append({
                "Model": model_name,
                "Configuration": config_name,
                "Weighted F1 Mean": weighted_scores.mean(),
                "Weighted F1 Std": weighted_scores.std(),
                "Weighted F1 95% CI": 1.96 * weighted_scores.std() / np.sqrt(len(weighted_scores)),
                "Macro F1 Mean": macro_scores.mean(),
                "Macro F1 Std": macro_scores.std(),
                "Macro F1 95% CI": 1.96 * macro_scores.std() / np.sqrt(len(macro_scores))
            })

        except Exception as e:
            print(f"  -> FAILED with error: {e}")


# Results table

ablation_df = pd.DataFrame(ablation_results)

if not ablation_df.empty:
    ablation_df = ablation_df.sort_values(
        by=["Model", "Weighted F1 Mean"],
        ascending=[True, False]
    )

    display(ablation_df.round(4))
else:
    print("No results were successfully collected.")

ablation_df.to_csv("src/ablation_results.csv", index=False)


Running ablation for: Logistic Regression
  -> Configuration: Structured-only
  -> Configuration: Text-only
  -> Configuration: Combined

Running ablation for: Linear SVM
  -> Configuration: Structured-only
  -> Configuration: Text-only
  -> Configuration: Combined

Running ablation for: Random Forest
  -> Configuration: Structured-only
  -> Configuration: Text-only
  -> Configuration: Combined

Running ablation for: Gradient Boosting
  -> Configuration: Structured-only
  -> Configuration: Text-only
  -> Configuration: Combined


,Model,Configuration,Weighted F1 Mean,Weighted F1 Std,Weighted F1 95% CI,Macro F1 Mean,Macro F1 Std,Macro F1 95% CI
11,Gradient Boosting,Combined,0.5841,0.0242,0.0212,0.5791,0.0189,0.0166
10,Gradient Boosting,Text-only,0.5822,0.0245,0.0215,0.5738,0.0228,0.0200
9,Gradient Boosting,Structured-only,0.3393,0.0089,0.0078,0.3183,0.0116,0.0102
5,Linear SVM,Combined,0.5437,0.0182,0.0160,0.5432,0.0148,0.0130
4,Linear SVM,Text-only,0.5368,0.0294,0.0257,0.5372,0.0319,0.0280
3,Linear SVM,Structured-only,0.3059,0.0185,0.0162,0.3010,0.0165,0.0145
1,Logistic Regression,Text-only,0.5453,0.0165,0.0145,0.5482,0.0141,0.0124
2,Logistic Regression,Combined,0.5395,0.0214,0.0188,0.5430,0.0184,0.0161
0,Logistic Regression,Structured-only,0.2884,0.0309,0.0271,0.2907,0.0267,0.0234
8,Random Forest,Combined,0.5897,0.0214,0.0188,0.5787,0.0297,0.0260


#### Ablation Study Results and Interpretation:

The ablation study shows that review text is much more predictive than structured features alone. Across all model types, the structured-only configuration produced the weakest performance, with weighted F1 scores around 0.29 to 0.34. In contrast, the text-only models performed much better, with weighted F1 scores around 0.54 to 0.58.

The highest mean result came from the Random Forest model using the combined configuration, with a weighted F1 score of 0.59 and a macro F1 score of 0.58. Gradient Boosting also performed strongly, with the combined configuration reaching about 0.584 weighted F1. However, the difference between Random Forest combined and Gradient Boosting combined is small, and their confidence intervals overlap substantially. Therefore, Random Forest should not be treated as a definitive winner.

The main conclusion is that review text contains the strongest signal for predicting customer ratings. Structured features alone are not sufficient, but they may add some value when combined with text, especially for the Random Forest model. Overall, using both text and structured features is justified for final model development, while the difference between the strongest models should be interpreted cautiously.

## 4.2 Model Exploration

Training four model types appropriate to the target.


In [11]:

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier

# Linear model: Logistic Regression
model_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    multi_class="multinomial",
    random_state=42
)

# Tree-based model: Random Forest
model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=None
)

# Boosted model: HistGradientBoosting with early stopping
model_hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

# Ensemble model: Voting Classifier
model_ensemble = VotingClassifier(
    estimators=[
        ("lr", model_lr),
        ("rf", model_rf),
        ("hgb", model_hgb)
    ],
    voting="soft"
)

exploration_models = {
    "Linear - Logistic Regression": {
        "model": model_lr,
        "hyperparameters": "max_iter=2000, class_weight=balanced, solver=lbfgs, multi_class=multinomial"
    },
    "Tree-based - Random Forest": {
        "model": model_rf,
        "hyperparameters": "n_estimators=200, max_depth=15, min_samples_split=5, min_samples_leaf=2, class_weight=balanced"
    },
    "Boosted - HistGradientBoosting": {
        "model": model_hgb,
        "hyperparameters": "max_iter=300, learning_rate=0.05, max_leaf_nodes=31, early_stopping=True, validation_fraction=0.1, n_iter_no_change=10"
    },
    "Ensemble - Voting Classifier": {
        "model": model_ensemble,
        "hyperparameters": "soft voting using Logistic Regression, Random Forest, and HistGradientBoosting"
    }
}

print("Evaluating models on the combined feature configuration...")

exploration_results = []

for model_name, model_info in exploration_models.items():
    print(f"\nTraining {model_name}...")

    pipe = Pipeline([
        ("preprocessor", combined_preprocessor),
        ("model", model_info["model"])
    ])

    try:
        scores = cross_validate(
            pipe,
            X,
            y,
            cv=cv_strategy,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False,
            error_score="raise"
        )

        weighted_scores = scores["test_weighted_f1"]
        macro_scores = scores["test_macro_f1"]

        exploration_results.append({
            "Model": model_name,
            "Main Hyperparameters": model_info["hyperparameters"],
            "Weighted F1 Mean": weighted_scores.mean(),
            "Weighted F1 Std": weighted_scores.std(),
            "Weighted F1 95% CI": 1.96 * weighted_scores.std() / np.sqrt(len(weighted_scores)),
            "Macro F1 Mean": macro_scores.mean(),
            "Macro F1 Std": macro_scores.std(),
            "Macro F1 95% CI": 1.96 * macro_scores.std() / np.sqrt(len(macro_scores))
        })

    except Exception as e:
        print(f"  -> FAILED with error: {e}")

exploration_df = pd.DataFrame(exploration_results)

if not exploration_df.empty:
    exploration_df = exploration_df.sort_values(
        by="Weighted F1 Mean",
        ascending=False
    )

    display(exploration_df.round(4))
else:
    print("No results generated.")

exploration_df.to_csv("src/exploration_df.csv", index=False)

Evaluating models on the combined feature configuration...

Training Linear - Logistic Regression...

Training Tree-based - Random Forest...

Training Boosted - HistGradientBoosting...

Training Ensemble - Voting Classifier...


,Model,Main Hyperparameters,Weighted F1 Mean,Weighted F1 Std,Weighted F1 95% CI,Macro F1 Mean,Macro F1 Std,Macro F1 95% CI
3,Ensemble - Voting Classifier,"soft voting using Logistic Regression, Random ...",0.6000,0.0204,0.0179,0.5982,0.0242,0.0212
1,Tree-based - Random Forest,"n_estimators=200, max_depth=15, min_samples_sp...",0.5927,0.0269,0.0236,0.5849,0.0306,0.0269
2,Boosted - HistGradientBoosting,"max_iter=300, learning_rate=0.05, max_leaf_nod...",0.5704,0.0211,0.0185,0.5652,0.0187,0.0164
0,Linear - Logistic Regression,"max_iter=2000, class_weight=balanced, solver=l...",0.5395,0.0214,0.0188,0.5430,0.0184,0.0161


#### Model Exploration Results and Interpretation:

The model exploration results show that the Ensemble, Voting Classifier performed best overall, with a weighted F1 score of 0.60 and a macro F1 score of 0.60. This means the ensemble model handled the multi-class rating prediction task better than the individual linear, tree-based, and boosted models.

The Random Forest model was the second-best model, with a weighted F1 score of 0.59 and a macro F1 score of 0.58. This is consistent with the ablation study, where Random Forest also performed strongly with the combined feature configuration.

The boosted HistGradientBoosting model achieved a weighted F1 score of 0.57, while Logistic Regression had the lowest performance among the explored models, with a weighted F1 score of 0.54. This suggests that the relationship between the combined text and structured features and the rating outcome is not fully linear.

Overall, the Voting Classifier is the strongest model from this exploration step. It likely performs best because it combines the strengths of Logistic Regression, Random Forest, and HistGradientBoosting instead of relying on one model type alone.